In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgb

In [4]:
booster = {}
booster['boosting_type'] = 'gbdt'
booster['objective'] = 'binary'
booster['learning_rate'] = 0.02
booster['num_leaves'] = 128
booster['max_depth'] = -1
booster['min_child_weight'] = 100
booster['max_bin'] = 1024
booster['subsample'] = 0.7
booster['subsample_freq'] = 1
booster['colsample_bytree'] = 0.5
booster['min_split_gain'] = 0
booster['nthread'] = 15
booster['verbose'] = 0
booster['metric'] = 'auc'

### English

In [3]:
feats = ['use_' + str(x) for x in range(512)]

In [4]:
train = pd.read_csv('../../data/process/english/train_english_embed.csv')
valid = pd.read_csv('../../data/process/english/valid_english_embed.csv')
test = pd.read_csv('../../data/process/english/test_english_embed.csv')

/usr/local/lib/python3.6/dist-packages/IPython/core/interactiveshell.py:3063: DtypeWarning: Columns (0) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [5]:
data = train.append(valid).append(test)
labels = [0] * len(train) + [1] * (len(valid) + len(test))
data['label'] = labels
data = data.sample(frac=1., random_state=2017).reset_index(drop=True)

In [6]:
data.shape

(945729, 519)

In [7]:
def crossFold(subset):
    train_data = data[data.index % 5 != subset].reset_index(drop=True)
    valid_data = data[data.index % 5 == subset].reset_index(drop=True)
    valid_index = valid_data[['id','source','lang']].copy()
    train_matrix = lgb.Dataset(train_data[feats], np.array(train_data['label']).reshape(-1,))
    valid_matrix = lgb.Dataset(valid_data[feats], np.array(valid_data['label']).reshape(-1,))
    params = {}
    params['params'] = booster
    params['train_set'] = train_matrix
    params['valid_sets'] = [train_matrix, valid_matrix]
    params['num_boost_round'] = 250
    params['early_stopping_rounds'] = 50
    params['verbose_eval'] = 50
    model = lgb.train(**params)
    valid_index['score'] = model.predict(valid_data[feats])
    return valid_index

In [8]:
valid_0 = crossFold(0)

Training until validation scores don't improve for 50 rounds
[50]	training's auc: 0.936846	valid_1's auc: 0.933356
[100]	training's auc: 0.945134	valid_1's auc: 0.940951
[150]	training's auc: 0.952176	valid_1's auc: 0.947163
[200]	training's auc: 0.957909	valid_1's auc: 0.952138
[250]	training's auc: 0.962541	valid_1's auc: 0.956073
Did not meet early stopping. Best iteration is:
[250]	training's auc: 0.962541	valid_1's auc: 0.956073


In [9]:
valid_1 = crossFold(1)

Training until validation scores don't improve for 50 rounds
[50]	training's auc: 0.937043	valid_1's auc: 0.931425
[100]	training's auc: 0.945408	valid_1's auc: 0.939163
[150]	training's auc: 0.95237	valid_1's auc: 0.945454
[200]	training's auc: 0.958035	valid_1's auc: 0.950499
[250]	training's auc: 0.962681	valid_1's auc: 0.95454
Did not meet early stopping. Best iteration is:
[250]	training's auc: 0.962681	valid_1's auc: 0.95454


In [10]:
valid_2 = crossFold(2)

Training until validation scores don't improve for 50 rounds
[50]	training's auc: 0.937371	valid_1's auc: 0.93258
[100]	training's auc: 0.945606	valid_1's auc: 0.939971
[150]	training's auc: 0.952413	valid_1's auc: 0.945969
[200]	training's auc: 0.958152	valid_1's auc: 0.950975
[250]	training's auc: 0.962781	valid_1's auc: 0.954892
Did not meet early stopping. Best iteration is:
[250]	training's auc: 0.962781	valid_1's auc: 0.954892


In [11]:
valid_3 = crossFold(3)

Training until validation scores don't improve for 50 rounds
[50]	training's auc: 0.937241	valid_1's auc: 0.931942
[100]	training's auc: 0.945495	valid_1's auc: 0.939392
[150]	training's auc: 0.952423	valid_1's auc: 0.945704
[200]	training's auc: 0.958128	valid_1's auc: 0.950782
[250]	training's auc: 0.96277	valid_1's auc: 0.954818
Did not meet early stopping. Best iteration is:
[250]	training's auc: 0.96277	valid_1's auc: 0.954818


In [12]:
valid_4 = crossFold(4)

Training until validation scores don't improve for 50 rounds
[50]	training's auc: 0.937081	valid_1's auc: 0.932236
[100]	training's auc: 0.945261	valid_1's auc: 0.939599
[150]	training's auc: 0.952231	valid_1's auc: 0.9458
[200]	training's auc: 0.957966	valid_1's auc: 0.950866
[250]	training's auc: 0.962597	valid_1's auc: 0.954864
Did not meet early stopping. Best iteration is:
[250]	training's auc: 0.962597	valid_1's auc: 0.954864


In [13]:
data = valid_0.append(valid_1).append(valid_2).append(valid_3).append(valid_4)

In [14]:
data.to_csv('../../data/process/english/adverse.csv', index=False)

In [15]:
data.shape

(945729, 4)

In [16]:
sum(data['score'] > 0.67)

59884

### Subtitle

In [17]:
feats = ['use_' + str(x) for x in range(512)]

In [18]:
train = pd.read_csv('../../data/process/subtitle/subtitle_embed.csv')
valid = pd.read_csv('../../data/process/foreign/valid_foreign_embed.csv')
test = pd.read_csv('../../data/process/foreign/test_foreign_embed.csv')
valid = valid[valid['original'] == 1]
test = test[test['original'] == 1]
valid = valid.drop('original', axis=1)
test = test.drop('original', axis=1)

In [19]:
data = train.append(valid).append(test)
labels = [0] * len(train) + [1] * (len(valid) + len(test))
data['label'] = labels
data = data.sample(frac=1., random_state=2017).reset_index(drop=True)

In [20]:
data.shape

(708138, 518)

In [21]:
def crossFold(subset):
    train_data = data[data.index % 5 != subset].reset_index(drop=True)
    valid_data = data[data.index % 5 == subset].reset_index(drop=True)
    valid_index = valid_data[['id','source','lang']].copy()
    train_matrix = lgb.Dataset(train_data[feats], np.array(train_data['label']).reshape(-1,))
    valid_matrix = lgb.Dataset(valid_data[feats], np.array(valid_data['label']).reshape(-1,))
    params = {}
    params['params'] = booster
    params['train_set'] = train_matrix
    params['valid_sets'] = [train_matrix, valid_matrix]
    params['num_boost_round'] = 250
    params['early_stopping_rounds'] = 50
    params['verbose_eval'] = 50
    model = lgb.train(**params)
    valid_index['score'] = model.predict(valid_data[feats])
    return valid_index

In [22]:
valid_0 = crossFold(0)

Training until validation scores don't improve for 50 rounds
[50]	training's auc: 0.989927	valid_1's auc: 0.98847
[100]	training's auc: 0.992299	valid_1's auc: 0.990752
[150]	training's auc: 0.993954	valid_1's auc: 0.992231
[200]	training's auc: 0.995221	valid_1's auc: 0.993433
[250]	training's auc: 0.996108	valid_1's auc: 0.994313
Did not meet early stopping. Best iteration is:
[250]	training's auc: 0.996108	valid_1's auc: 0.994313


In [23]:
valid_1 = crossFold(1)

Training until validation scores don't improve for 50 rounds
[50]	training's auc: 0.989967	valid_1's auc: 0.987988
[100]	training's auc: 0.992393	valid_1's auc: 0.990332
[150]	training's auc: 0.994003	valid_1's auc: 0.991925
[200]	training's auc: 0.995217	valid_1's auc: 0.993197
[250]	training's auc: 0.99608	valid_1's auc: 0.994123
Did not meet early stopping. Best iteration is:
[250]	training's auc: 0.99608	valid_1's auc: 0.994123


In [24]:
valid_2 = crossFold(2)

Training until validation scores don't improve for 50 rounds
[50]	training's auc: 0.990006	valid_1's auc: 0.988184
[100]	training's auc: 0.992379	valid_1's auc: 0.990437
[150]	training's auc: 0.993979	valid_1's auc: 0.992047
[200]	training's auc: 0.995226	valid_1's auc: 0.993279
[250]	training's auc: 0.996097	valid_1's auc: 0.994142
Did not meet early stopping. Best iteration is:
[250]	training's auc: 0.996097	valid_1's auc: 0.994142


In [25]:
valid_3 = crossFold(3)

Training until validation scores don't improve for 50 rounds
[50]	training's auc: 0.990028	valid_1's auc: 0.987988
[100]	training's auc: 0.992394	valid_1's auc: 0.99032
[150]	training's auc: 0.994024	valid_1's auc: 0.991849
[200]	training's auc: 0.995252	valid_1's auc: 0.993098
[250]	training's auc: 0.996125	valid_1's auc: 0.994011
Did not meet early stopping. Best iteration is:
[250]	training's auc: 0.996125	valid_1's auc: 0.994011


In [26]:
valid_4 = crossFold(4)

Training until validation scores don't improve for 50 rounds
[50]	training's auc: 0.990045	valid_1's auc: 0.988257
[100]	training's auc: 0.992401	valid_1's auc: 0.99046
[150]	training's auc: 0.994005	valid_1's auc: 0.991981
[200]	training's auc: 0.995225	valid_1's auc: 0.993233
[250]	training's auc: 0.996098	valid_1's auc: 0.994101
Did not meet early stopping. Best iteration is:
[250]	training's auc: 0.996098	valid_1's auc: 0.994101


In [27]:
data = valid_0.append(valid_1).append(valid_2).append(valid_3).append(valid_4)

In [28]:
data.to_csv('../../data/process/subtitle/adverse.csv', index=False)

In [29]:
data.shape

(708138, 4)

In [30]:
sum(data['score'] > 0.9)

44310

### Translation

In [5]:
feats = ['use_' + str(x) for x in range(512)]

In [6]:
dtypes = {x: np.float16 for x in feats}

In [7]:
train = pd.read_csv('../../data/process/foreign/train_foreign_embed.csv', dtype=dtypes)
train[feats] = train[feats].astype(np.float16)
valid = pd.read_csv('../../data/process/foreign/valid_foreign_embed.csv', dtype=dtypes)
valid[feats] = valid[feats].astype(np.float16)
test = pd.read_csv('../../data/process/foreign/test_foreign_embed.csv', dtype=dtypes)
test[feats] = test[feats].astype(np.float16)
valid = valid[valid['original'] == 1]
test = test[test['original'] == 1]
valid = valid.drop('original', axis=1)
test = test.drop('original', axis=1)

/usr/local/lib/python3.6/dist-packages/IPython/core/interactiveshell.py:3063: DtypeWarning: Columns (0) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [8]:
data = train.append(valid).append(test)
labels = [0] * len(train) + [1] * (len(valid) + len(test))
data['label'] = labels
train, valid, test = None, None, None
data = data.sample(frac=1., random_state=2017).reset_index(drop=True)

In [9]:
data.shape

(2970162, 518)

In [10]:
def crossFold(subset):
    train_data = data[data.index % 5 != subset].reset_index(drop=True)
    valid_data = data[data.index % 5 == subset].reset_index(drop=True)
    valid_index = valid_data[['id','source','lang']].copy()
    train_matrix = lgb.Dataset(train_data[feats], np.array(train_data['label']).reshape(-1,))
    valid_matrix = lgb.Dataset(valid_data[feats], np.array(valid_data['label']).reshape(-1,))
    params = {}
    params['params'] = booster
    params['train_set'] = train_matrix
    params['valid_sets'] = [train_matrix, valid_matrix]
    params['num_boost_round'] = 250
    params['early_stopping_rounds'] = 50
    params['verbose_eval'] = 50
    model = lgb.train(**params)
    valid_index['score'] = model.predict(valid_data[feats])
    return valid_index

In [ ]:
valid_0 = crossFold(0)

Training until validation scores don't improve for 50 rounds
[50]	training's auc: 0.91584	valid_1's auc: 0.908466
[100]	training's auc: 0.927853	valid_1's auc: 0.919129
[150]	training's auc: 0.937854	valid_1's auc: 0.927926
[200]	training's auc: 0.945797	valid_1's auc: 0.934838


In [12]:
valid_1 = crossFold(1)

Training until validation scores don't improve for 50 rounds
[50]	training's auc: 0.916291	valid_1's auc: 0.906844
[100]	training's auc: 0.927957	valid_1's auc: 0.917374
[150]	training's auc: 0.937912	valid_1's auc: 0.926308
[200]	training's auc: 0.94591	valid_1's auc: 0.933487
[250]	training's auc: 0.95216	valid_1's auc: 0.938769
Did not meet early stopping. Best iteration is:
[250]	training's auc: 0.95216	valid_1's auc: 0.938769


In [13]:
valid_2 = crossFold(2)

Training until validation scores don't improve for 50 rounds
[50]	training's auc: 0.915846	valid_1's auc: 0.907871
[100]	training's auc: 0.927692	valid_1's auc: 0.918264
[150]	training's auc: 0.937797	valid_1's auc: 0.927011
[200]	training's auc: 0.945725	valid_1's auc: 0.933944
[250]	training's auc: 0.952031	valid_1's auc: 0.939294
Did not meet early stopping. Best iteration is:
[250]	training's auc: 0.952031	valid_1's auc: 0.939294


In [14]:
valid_3 = crossFold(3)

Training until validation scores don't improve for 50 rounds
[50]	training's auc: 0.916228	valid_1's auc: 0.905693
[100]	training's auc: 0.92804	valid_1's auc: 0.916384
[150]	training's auc: 0.937945	valid_1's auc: 0.925237
[200]	training's auc: 0.945864	valid_1's auc: 0.932186
[250]	training's auc: 0.95217	valid_1's auc: 0.937658
Did not meet early stopping. Best iteration is:
[250]	training's auc: 0.95217	valid_1's auc: 0.937658


In [15]:
valid_4 = crossFold(4)

Training until validation scores don't improve for 50 rounds
[50]	training's auc: 0.915731	valid_1's auc: 0.907044
[100]	training's auc: 0.927811	valid_1's auc: 0.917698
[150]	training's auc: 0.937774	valid_1's auc: 0.926526
[200]	training's auc: 0.945692	valid_1's auc: 0.933467
[250]	training's auc: 0.952079	valid_1's auc: 0.938936
Did not meet early stopping. Best iteration is:
[250]	training's auc: 0.952079	valid_1's auc: 0.938936


In [16]:
data = valid_0.append(valid_1).append(valid_2).append(valid_3).append(valid_4)

In [17]:
data.to_csv('../../data/process/foreign/adverse.csv', index=False)

In [18]:
data.shape

(2970162, 4)

In [19]:
sum(data['score'] > 0.67)

3972

### Valid

In [20]:
feats = ['use_' + str(x) for x in range(512)]

In [21]:
train = pd.read_csv('../../data/process/foreign/valid_foreign_embed.csv')
train = train[train['original'] == 1]
valid = pd.read_csv('../../data/process/foreign/valid_foreign_embed.csv')
valid = valid[valid['original'] == 0]
test = pd.read_csv('../../data/process/foreign/test_foreign_embed.csv')
test = test[test['original'] == 0]

In [22]:
data = train.append(valid).append(test)
data = data.reset_index(drop=True)
labels = [1] * len(train) + [0] * (len(valid) + len(test))
data['label'] = labels
data = data.sample(frac=1., random_state=2017).reset_index(drop=True)

In [23]:
data.shape

(366489, 519)

In [24]:
def crossFold(subset):
    train_data = data[data.index % 5 != subset].reset_index(drop=True)
    valid_data = data[data.index % 5 == subset].reset_index(drop=True)
    valid_index = valid_data[['id','source','lang']].copy()
    train_matrix = lgb.Dataset(train_data[feats], np.array(train_data['label']).reshape(-1,))
    valid_matrix = lgb.Dataset(valid_data[feats], np.array(valid_data['label']).reshape(-1,))
    params = {}
    params['params'] = booster
    params['train_set'] = train_matrix
    params['valid_sets'] = [train_matrix, valid_matrix]
    params['num_boost_round'] = 250
    params['early_stopping_rounds'] = 50
    params['verbose_eval'] = 50
    model = lgb.train(**params)
    valid_index['score'] = model.predict(valid_data[feats])
    return valid_index

In [25]:
valid_0 = crossFold(0)

Training until validation scores don't improve for 50 rounds
[50]	training's auc: 0.828532	valid_1's auc: 0.773919
[100]	training's auc: 0.855912	valid_1's auc: 0.791809
[150]	training's auc: 0.877404	valid_1's auc: 0.806805
[200]	training's auc: 0.893465	valid_1's auc: 0.817105
[250]	training's auc: 0.906488	valid_1's auc: 0.825187
Did not meet early stopping. Best iteration is:
[250]	training's auc: 0.906488	valid_1's auc: 0.825187


In [26]:
valid_1 = crossFold(1)

Training until validation scores don't improve for 50 rounds
[50]	training's auc: 0.827615	valid_1's auc: 0.782759
[100]	training's auc: 0.85615	valid_1's auc: 0.804687
[150]	training's auc: 0.876292	valid_1's auc: 0.818635
[200]	training's auc: 0.892658	valid_1's auc: 0.829661
[250]	training's auc: 0.905583	valid_1's auc: 0.83802
Did not meet early stopping. Best iteration is:
[250]	training's auc: 0.905583	valid_1's auc: 0.83802


In [27]:
valid_2 = crossFold(2)

Training until validation scores don't improve for 50 rounds
[50]	training's auc: 0.829098	valid_1's auc: 0.76887
[100]	training's auc: 0.857848	valid_1's auc: 0.79371
[150]	training's auc: 0.877325	valid_1's auc: 0.808057
[200]	training's auc: 0.89287	valid_1's auc: 0.819094
[250]	training's auc: 0.905758	valid_1's auc: 0.827408
Did not meet early stopping. Best iteration is:
[250]	training's auc: 0.905758	valid_1's auc: 0.827408


In [28]:
valid_3 = crossFold(3)

Training until validation scores don't improve for 50 rounds
[50]	training's auc: 0.826189	valid_1's auc: 0.772351
[100]	training's auc: 0.856434	valid_1's auc: 0.795509
[150]	training's auc: 0.876317	valid_1's auc: 0.810133
[200]	training's auc: 0.89292	valid_1's auc: 0.820306
[250]	training's auc: 0.905642	valid_1's auc: 0.829216
Did not meet early stopping. Best iteration is:
[250]	training's auc: 0.905642	valid_1's auc: 0.829216


In [29]:
valid_4 = crossFold(4)

Training until validation scores don't improve for 50 rounds
[50]	training's auc: 0.82377	valid_1's auc: 0.780507
[100]	training's auc: 0.853386	valid_1's auc: 0.802423
[150]	training's auc: 0.874352	valid_1's auc: 0.818009
[200]	training's auc: 0.890769	valid_1's auc: 0.830504
[250]	training's auc: 0.903965	valid_1's auc: 0.839309
Did not meet early stopping. Best iteration is:
[250]	training's auc: 0.903965	valid_1's auc: 0.839309


In [30]:
data = valid_0.append(valid_1).append(valid_2).append(valid_3).append(valid_4)

In [31]:
data.to_csv('../../data/process/foreign/adverse_valid.csv', index=False)

In [32]:
data.shape

(366489, 4)

In [33]:
sum(data['score'] >= 0.1)

2465